In [1]:
from pyspark.sql import SparkSession # type: ignore

spark = SparkSession.builder \
    .appName("SparkCourse") \
    .master("local[*]") \
    .config("spark.sql.warehouse.dir", "/home/jovyan/work/setup/spark-warehouse") \
    .config("spark.hadoop.javax.jdo.option.ConnectionURL",
            "jdbc:derby:/home/jovyan/work/metastore_db;create=true") \
    .config("spark.hadoop.javax.jdo.option.ConnectionDriverName",
            "org.apache.derby.jdbc.EmbeddedDriver") \
    .enableHiveSupport() \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.0


In [2]:
"""
Other Types of joins
    Natural Join - Automatically create join criteria on the same column names (Applies to Inner and Outer Joins)
    Cross Join - Join without any join criteria (all possible combinations)
    Self Join - Join a table with itself (Applies to Inner, Outer, and Cross Joins)
    Semi Join - Take records from the left side when it matches with the right side (Correlated EXISTS)
    Anti Join - Take records from the left side when it doesn not match with the right side (Correlated NOT EXISTS)
"""

'\nOther Types of joins\n    Natural Join - Automatically create join criteria on the same column names (Applies to Inner and Outer Joins)\n    Cross Join - Join without any join criteria (all possible combinations)\n    Self Join - Join a table with itself (Applies to Inner, Outer, and Cross Joins)\n    Semi Join - Take records from the left side when it matches with the right side (Correlated EXISTS)\n    Anti Join - Take records from the left side when it doesn not match with the right side (Correlated NOT EXISTS)\n'

In [3]:
spark.sql("SELECT * FROM spark_db.members").show()
spark.sql("SELECT * FROM spark_db.bookings").show()
spark.sql("SELECT * FROM spark_db.facilities").show()

+-----+---------+---------+--------------------+-------+--------------+-------------+-------------------+
|memid|  surname|firstname|             address|zipcode|     telephone|recommendedby|           joindate|
+-----+---------+---------+--------------------+-------+--------------+-------------+-------------------+
|    0|    GUEST|    GUEST|               GUEST|      0|(000) 000-0000|         NULL|2022-07-01 00:00:00|
|    1|    Smith|   Darren|8 Bloomsbury Clos...|   4321|  555-555-5555|         NULL|2022-07-02 12:02:05|
|    2|    Smith|    Tracy|8 Bloomsbury Clos...|   4321|  555-555-5555|         NULL|2022-07-02 12:08:23|
|    3|   Rownam|      Tim|23 Highway Way, B...|  23423|(844) 693-0723|         NULL|2022-07-03 09:32:15|
|    4| Joplette|   Janice|20 Crossing Road,...|    234|(833) 942-4710|            1|2022-07-03 10:25:05|
|    5|  Butters|   Gerald|1065 Huntingdon A...|  56754|(844) 078-4130|            1|2022-07-09 10:44:09|
|    6|    Tracy|   Burton|3 Tunisia Drive, ..

In [4]:
"""
Q1. Show me a facility bookings report as the following. (Prefer Natural Join)

member_id | first_name | last_name | facility_name | slots | booking_amount | start_time
--------------------------------------------------------------------------------------------
The report must meet the following criteria.

    Facility bookings made by a person whose last name is Smith
    He has booked more than 5 slots in a single booking
    Report should be sorted by first name of the member in ascending order and booking amount in descending order
"""
from pyspark.sql.functions import col # type: ignore

bookings_df = spark.table("spark_db.bookings").filter(col("slots") > 5).alias("b")
members_df = spark.table("spark_db.members").filter(col("surname") == 'Smith').alias("m")
facilities_df = spark.table("spark_db.facilities").alias("f")

bmf_df = bookings_df.join(members_df, "memid", "inner")\
                    .join(facilities_df, "facid") # no join condition only the column name from both the tables that needs to be used for the join

reports_df = bmf_df.select(
        col("memid").alias("member_id"),
        col("firstname").alias("first_name"),
        col("surname").alias("last_name"),
        col("fac_name").alias("facility_name"),
        col("slots"),
        (col("slots") * col("membercost")).alias("booking_amount"),
        col("starttime").alias("start_time")
    )\
    .orderBy(col("firstname"), col("booking_amount").desc())
reports_df.show()

+---------+----------+---------+---------------+-----+--------------+-------------------+
|member_id|first_name|last_name|  facility_name|slots|booking_amount|         start_time|
+---------+----------+---------+---------------+-----+--------------+-------------------+
|        1|    Darren|    Smith|Badminton Court|    6|             0|2022-07-09 09:00:00|
|        1|    Darren|    Smith|Badminton Court|    6|             0|2022-07-27 12:00:00|
|        1|    Darren|    Smith|Badminton Court|    6|             0|2022-07-29 12:00:00|
|        1|    Darren|    Smith|Badminton Court|    6|             0|2022-08-01 09:30:00|
|        1|    Darren|    Smith|Badminton Court|    6|             0|2022-08-07 09:00:00|
|        1|    Darren|    Smith|Badminton Court|    6|             0|2022-08-20 15:00:00|
|        1|    Darren|    Smith|Badminton Court|    9|             0|2022-08-28 13:30:00|
|        1|    Darren|    Smith|Badminton Court|    6|             0|2022-09-07 14:00:00|
|        1

In [5]:
"""
Q2. Prepare a member bookings report as the following (Prefer Natural Join)

booking_id | facility_name | slots | first_name | last_name | address
Ensure the following

    Consider only regular memebrs (not guest) and direct members(not recomended by any other member)
    Consider only bookings for more than 8 hours
    Ensure all regular and direct members are listed even if they have no 8 hour bookings
    Ensure all 8 hour bookings are listed even if they are not made by regular and direct members
    Sort the report by slots and first name in ascending order
"""



'\nQ2. Prepare a member bookings report as the following (Prefer Natural Join)\n\nbooking_id | facility_name | slots | first_name | last_name | address\nEnsure the following\n\n    Consider only regular memebrs (not guest) and direct members(not recomended by any other member)\n    Consider only bookings for more than 8 hours\n    Ensure all regular and direct members are listed even if they have no 8 hour bookings\n    Ensure all 8 hour bookings are listed even if they are not made by regular and direct members\n    Sort the report by slots and first name in ascending order\n'

In [6]:
"""
Q3. How many bookings are possible when each member is booking a facility exactly once in a month?
Show all possible combinations
"""

members_df = spark.table("spark_db.members").filter(col("memid") > 0)
facilities_df = spark.table("spark_db.facilities")

reports_df = members_df.crossJoin(facilities_df).select( # no criteria for join as this is a cross join
    col("firstname"), col("surname"), col("fac_name")
) 

reports_df.show()

+---------+-------+---------------+
|firstname|surname|       fac_name|
+---------+-------+---------------+
|   Darren|  Smith| Tennis Court 1|
|   Darren|  Smith| Tennis Court 2|
|   Darren|  Smith|Badminton Court|
|   Darren|  Smith|   Table Tennis|
|   Darren|  Smith| Massage Room 1|
|   Darren|  Smith| Massage Room 2|
|   Darren|  Smith|   Squash Court|
|   Darren|  Smith|  Snooker Table|
|   Darren|  Smith|     Pool Table|
|    Tracy|  Smith| Tennis Court 1|
|    Tracy|  Smith| Tennis Court 2|
|    Tracy|  Smith|Badminton Court|
|    Tracy|  Smith|   Table Tennis|
|    Tracy|  Smith| Massage Room 1|
|    Tracy|  Smith| Massage Room 2|
|    Tracy|  Smith|   Squash Court|
|    Tracy|  Smith|  Snooker Table|
|    Tracy|  Smith|     Pool Table|
|      Tim| Rownam| Tennis Court 1|
|      Tim| Rownam| Tennis Court 2|
+---------+-------+---------------+
only showing top 20 rows



In [7]:
"""
Q4. Prepare a report for members and who recomended them as the following

member_id | Member Name | Recommended By
--------------------------------------------
"""
from pyspark.sql.functions import concat_ws # type: ignore

members_self_join = col("m.recommendedby") == col("r.memid")

result_df = members_df.alias("m").join(members_df.alias("r"), members_self_join, "inner")\
                                .select(
                                    col("m.memid"),
                                    concat_ws(" ", col("m.firstname"), col("m.surname")).alias("Member Name"),
                                    concat_ws(" ", col("r.firstname"), col("r.surname")).alias("Recommended By")
                                )
result_df.show()

+-----+--------------------+---------------+
|memid|         Member Name| Recommended By|
+-----+--------------------+---------------+
|    4|     Janice Joplette|   Darren Smith|
|    5|      Gerald Butters|   Darren Smith|
|    7|          Nancy Dare|Janice Joplette|
|    8|          Tim Boothe|     Tim Rownam|
|    9|     Ponder Stibbons|   Burton Tracy|
|   10|        Charles Owen|   Darren Smith|
|   11|         David Jones|Janice Joplette|
|   12|          Anne Baker|Ponder Stibbons|
|   14|          Jack Smith|   Darren Smith|
|   15|      Florence Bader|Ponder Stibbons|
|   16|       Timothy Baker| Jemima Farrell|
|   17|        David Pinker| Jemima Farrell|
|   20|     Matthew Genting| Gerald Butters|
|   21|      Anna Mackenzie|   Darren Smith|
|   22|         Joan Coplin|  Timothy Baker|
|   24|    Ramnaresh Sarwin| Florence Bader|
|   26|       Douglas Jones|    David Jones|
|   27|    Henrietta Rumney|Matthew Genting|
|   29|Henry Worthington...|    Tracy Smith|
|   30|   

In [10]:
"""
Q5. Prepare a list of members who made at least one booking. (Use SEMI Join)

member_id | first_name | last_name | address
-----------------------------------------------
"""
members_df = spark.table("spark_db.members").filter(col("memid") > 0).alias("m")
bookings_df = spark.table("spark_db.bookings").alias("b")

members_bookings_join = col("m.memid") == col("b.memid")

report_df = members_df.join(bookings_df, members_bookings_join, "left_semi")\
                    .select(
                        col("m.memid"),
                        col("m.firstname"),
                        col("m.surname"),
                        col("m.address")
                    )
report_df.show()

+-----+---------+---------+--------------------+
|memid|firstname|  surname|             address|
+-----+---------+---------+--------------------+
|    1|   Darren|    Smith|8 Bloomsbury Clos...|
|    2|    Tracy|    Smith|8 Bloomsbury Clos...|
|    3|      Tim|   Rownam|23 Highway Way, B...|
|    4|   Janice| Joplette|20 Crossing Road,...|
|    5|   Gerald|  Butters|1065 Huntingdon A...|
|    6|   Burton|    Tracy|3 Tunisia Drive, ...|
|    7|    Nancy|     Dare|6 Hunting Lodge W...|
|    8|      Tim|   Boothe|3 Bloomsbury Clos...|
|    9|   Ponder| Stibbons|5 Dragons Way, Wi...|
|   10|  Charles|     Owen|52 Cheshire Grove...|
|   11|    David|    Jones|976 Gnats Close, ...|
|   12|     Anne|    Baker|55 Powdery Street...|
|   13|   Jemima|  Farrell|103 Firth Avenue,...|
|   14|     Jack|    Smith|252 Binkington Wa...|
|   15| Florence|    Bader|264 Ursula Drive,...|
|   16|  Timothy|    Baker|329 James Street,...|
|   17|    David|   Pinker|5 Impreza Road, B...|
|   20|  Matthew|  G

In [11]:
"""

Q6. Prepare a list of members who never made any bookings. (Use ANTI Join)

member_id | first_name | last_name | address
-----------------------------------------------
"""
members_df = spark.table("spark_db.members").filter(col("memid") > 0).alias("m")
bookings_df = spark.table("spark_db.bookings").alias("b")

members_bookings_join = col("m.memid") == col("b.memid")

report_df = members_df.join(bookings_df, members_bookings_join, "left_anti")\
                    .select(
                        col("m.memid"),
                        col("m.firstname"),
                        col("m.surname"),
                        col("m.address")
                    )
report_df.show()

+-----+---------+-------+--------------------+
|memid|firstname|surname|             address|
+-----+---------+-------+--------------------+
|   37|   Darren|  Smith|3 Funktown, Denzi...|
+-----+---------+-------+--------------------+

